# HR Governance ETL Validation

This notebook executes the synthetic-data contracts, temporal policy transform, Four-Fifths calculation, guardrail certification, and object-storage plan without a model key or network call. External pgvector and MinIO checks stay explicitly gated.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

cwd = Path.cwd().resolve()
if (cwd / 'backend').is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / 'backend').is_dir():
    REPO_ROOT = cwd.parent
else:
    raise RuntimeError('Run this notebook from the repository root or notebooks directory')
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

## Extract and source-contract checks

In [ ]:
from scripts.generate_hackathon_datasets import DEFAULT_OUT, generate_all

DATA_ROOT = DEFAULT_OUT
if not (DATA_ROOT / 'README.generated.json').exists():
    generate_all(DATA_ROOT, ats_records=10_000, guardrail_questions=500, resumes=1_000, seed=42)

policy_rows = read_csv(DATA_ROOT / 'poisoned_policies' / 'synthetic_policy_manifest.csv')
ats_rows = read_csv(DATA_ROOT / 'adverse_impact_ats' / 'synthetic_ats_hiring_data.csv')
guardrail_rows = read_csv(DATA_ROOT / 'pii_injection_stress' / 'synthetic_hr_guardrail_questions.csv')
resume_rows = read_csv(DATA_ROOT / 'resume_pdf_dump' / 'resume_manifest.csv')
batch_rows = [line for line in (DATA_ROOT / 'resume_pdf_dump' / 'fireworks_resume_vision_batch_manifest.jsonl').read_text(encoding='utf-8').splitlines() if line]

dataset_counts = {
    'policy_pdfs': len(list((DATA_ROOT / 'poisoned_policies').glob('*.pdf'))),
    'policy_manifest_rows': len(policy_rows),
    'ats_rows': len(ats_rows),
    'guardrail_rows': len(guardrail_rows),
    'resume_pdfs': len(list((DATA_ROOT / 'resume_pdf_dump' / 'pdfs').glob('*.pdf'))),
    'resume_manifest_rows': len(resume_rows),
    'batch_rows': len(batch_rows),
}
assert dataset_counts == {
    'policy_pdfs': 40,
    'policy_manifest_rows': 40,
    'ats_rows': 10_000,
    'guardrail_rows': 500,
    'resume_pdfs': 1_000,
    'resume_manifest_rows': 1_000,
    'batch_rows': 1_000,
}
dataset_counts

## Transform: temporal policy metadata

In [ ]:
from pipelines.ingestion import ingest_pdf
from services.rag import _apply_active_policy_version_filter

old_path = DATA_ROOT / 'poisoned_policies' / 'policy_pto_2023.pdf'
new_path = DATA_ROOT / 'poisoned_policies' / 'policy_pto_2024.pdf'
old_chunk = ingest_pdf(str(old_path), doc_id='policy_pto_2023')[0]
new_chunk = ingest_pdf(str(new_path), doc_id='policy_pto_2024')[0]
assert old_chunk['metadata']['status'] == 'expired'
assert new_chunk['metadata']['status'] == 'active'
assert new_chunk['metadata']['effective_date'] == '2024-01-01'

candidates = [
    {**old_chunk, 'score': 0.99},
    {**new_chunk, 'score': 0.72},
]
eligible = _apply_active_policy_version_filter(candidates)
assert [item['doc_id'] for item in eligible] == ['policy_pto_2024']
{'winner': eligible[0]['doc_id'], 'selection_basis': 'effective_date/status metadata'}

## Transform: Four-Fifths calculation

In [ ]:
from core.bias_audit import audit_hiring_csv

bias_audit = audit_hiring_csv(DATA_ROOT / 'adverse_impact_ats' / 'synthetic_ats_hiring_data.csv')
race_audit = next(item for item in bias_audit['dimensions'] if item['dimension'] == 'race')
group_rates = {item['group']: item['selection_rate'] for item in race_audit['groups']}
black_white_ratio = round(group_rates['Black'] / group_rates['White'], 4)
dimension_ratio = round(min(group_rates.values()) / max(group_rates.values()), 4)
assert dimension_ratio == race_audit['adverse_impact_ratio']
assert race_audit['adverse_impact_ratio'] < 0.80
assert bias_audit['violates_four_fifths_rule'] is True
{
    'selection_rates': group_rates,
    'black_vs_white_ratio': black_white_ratio,
    'dimension_ratio': dimension_ratio,
    'threshold': 0.80,
    'interpretation': 'review signal, not a legal conclusion',
}

## Load/probe and blocking certification

In [ ]:
certify_run = subprocess.run(
    [
        sys.executable,
        '-m',
        'scripts.certify_hackathon_synthetic_data',
        '--no-generate',
        '--out-dir',
        str(DATA_ROOT),
    ],
    cwd=BACKEND_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
certification = json.loads(certify_run.stdout)
assert certification['ok'] is True
assert certification['passed_count'] == certification['gate_count'] == 5
assert {gate['name'] for gate in certification['gates']} == {
    'poisoned_policy_versioning',
    'four_fifths_bias_audit',
    'pii_redaction_and_injection_block',
    'resume_pdf_batch_manifest',
    'resume_object_storage_plan',
}
certification

## Optional external integration gates

Set `HRCC_RUN_EXTERNAL_ETL=1` only when Postgres/pgvector and MinIO credentials are configured. A skipped gate remains `live_gated`, never passed.

In [ ]:
if os.getenv('HRCC_RUN_EXTERNAL_ETL') == '1':
    pgvector_run = subprocess.run(
        [sys.executable, '-m', 'scripts.load_poisoned_policies', '--deterministic', '--require-pgvector'],
        cwd=BACKEND_ROOT, check=True, capture_output=True, text=True, env=os.environ.copy(),
    )
    minio_run = subprocess.run(
        [sys.executable, '-m', 'scripts.upload_resume_dump', '--upload'],
        cwd=BACKEND_ROOT, check=True, capture_output=True, text=True, env=os.environ.copy(),
    )
    external_evidence = {
        'pgvector': json.loads(pgvector_run.stdout),
        'minio': json.loads(minio_run.stdout),
    }
else:
    external_evidence = {
        'pgvector': {'status': 'live_gated'},
        'minio': {'status': 'live_gated'},
    }
external_evidence